<a href="https://colab.research.google.com/github/sanskriti14/churn-predictor/blob/main/notebooks/churn_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Churn Prediction: Training Pipeline
This notebook contains the data preparation, feature engineering, and model training steps for the churn prediction system.

In [6]:
import pandas as pd
import numpy as np

# Set a random seed so the results are reproducible
np.random.seed(42)

# Generate 200 fake customers
n_customers = 200
account_age = np.random.randint(1, 72, n_customers)
monthly_charges = np.round(np.random.uniform(20.0, 180.0, n_customers), 2)
total_tickets = np.random.randint(0, 15, n_customers)
membership_type = np.random.randint(0, 3, n_customers) # 0: Basic, 1: Standard, 2: Premium

# Create a realistic math rule for churn:
# More support tickets and higher charges = much more likely to churn (1)
churn_chance = (total_tickets * 0.15) + (monthly_charges / 300) - (account_age / 150)
churned = (churn_chance > 0.4).astype(int)

# Package into a clean DataFrame
df_large = pd.DataFrame({
    "account_age_months": account_age,
    "monthly_charges": monthly_charges,
    "total_tickets": total_tickets,
    "membership_type": membership_type,
    "churned": churned
})

# Save this as your new dataset!
df_large.to_csv("data/customers.csv", index=False)

In [7]:
import os
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Load Data from sandbox file system
df = pd.read_csv("data/customers.csv")

# 2. Feature Engineering
membership_mapping = {"Basic": 1, "Standard": 2, "Premium": 3}
df['membership_encoded'] = df['membership_type'].map(membership_mapping)

features = ['account_age_months', 'monthly_charges', 'total_tickets', 'membership_encoded']
X = df[features]
y = df['churned']

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 4. Train Model
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

# 5. Evaluate
predictions = model.predict(X_test)
print(f"✅ Model Test Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%")

# 6. Save Model Artifact to sandbox
os.makedirs("models", exist_ok=True)
with open("models/churn_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("💾 Model artifact successfully saved to models/churn_model.pkl")

✅ Model Test Accuracy: 93.33%
💾 Model artifact successfully saved to models/churn_model.pkl


In [8]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. GENERATE THE PREDICTIONS FIRST (This creates 'y_pred')
# (Note: Change 'model' to 'rf_model' or whatever your model variable is named)
y_pred = model.predict(X_test)

# 2. Print the 2x2 grid
print(confusion_matrix(y_test, y_pred))

# 3. Print Accuracy, Precision, and Recall scores
print(classification_report(y_test, y_pred))

[[ 8  3]
 [ 1 48]]
              precision    recall  f1-score   support

           0       0.89      0.73      0.80        11
           1       0.94      0.98      0.96        49

    accuracy                           0.93        60
   macro avg       0.92      0.85      0.88        60
weighted avg       0.93      0.93      0.93        60

